In [1]:
import requests
import pandas as pd

# Teste: busca preço do Bitcoin e Ethereum agora
url = "https://api.coingecko.com/api/v3/simple/price"
params = {
    "ids": "bitcoin,ethereum",
    "vs_currencies": "usd,brl"
}

response = requests.get(url, params=params, timeout=10)
print("Status:", response.status_code)
print(response.json())

Status: 200
{'bitcoin': {'usd': 73168, 'brl': 366694}, 'ethereum': {'usd': 2250.58, 'brl': 11279.23}}


In [2]:
# o response.json() retorna um dicionário Python
# vamos explorar ele direito

dados_brutos = response.json()

print("Tipo do objeto:", type(dados_brutos))
print("Chaves:", list(dados_brutos.keys()))
print()

# acessando valores específicos
print("Bitcoin em USD:", dados_brutos["bitcoin"]["usd"])
print("Bitcoin em BRL:", dados_brutos["bitcoin"]["brl"])
print("Ethereum em USD:", dados_brutos["ethereum"]["usd"])

Tipo do objeto: <class 'dict'>
Chaves: ['bitcoin', 'ethereum']

Bitcoin em USD: 73168
Bitcoin em BRL: 366694
Ethereum em USD: 2250.58


In [3]:
import time

def buscar_moedas(paginas=2):
    """
    Busca as top moedas do mercado com paginação.
    paginas=2 → busca 500 moedas (250 por página)
    """
    
    BASE_URL = "https://api.coingecko.com/api/v3/coins/markets"
    todas_moedas = []
    
    for pagina in range(1, paginas + 1):
        print(f"Buscando página {pagina}...")
        
        params = {
            "vs_currency": "brl",
            "order": "market_cap_desc",  # ordenado por valor de mercado
            "per_page": 50,              # 50 por página (máximo free: 250)
            "page": pagina,
            "sparkline": "false"
        }
        
        response = requests.get(BASE_URL, params=params, timeout=30)
        response.raise_for_status()  # erro automático se != 200
        
        pagina_dados = response.json()
        
        if not pagina_dados:          # página vazia = acabou
            print("Sem mais dados.")
            break
        
        todas_moedas.extend(pagina_dados)
        print(f"  → {len(pagina_dados)} moedas | Total: {len(todas_moedas)}")
        
        time.sleep(1.5)               # respeita o rate limit
    
    return todas_moedas


# executa — busca 2 páginas = 100 moedas
moedas = buscar_moedas(paginas=2)
print(f"\nColeta finalizada! {len(moedas)} moedas coletadas.")

Buscando página 1...
  → 50 moedas | Total: 50
Buscando página 2...
  → 50 moedas | Total: 100

Coleta finalizada! 100 moedas coletadas.


In [4]:
# converte a lista de dicionários em um DataFrame
df = pd.DataFrame(moedas)

# seleciona só as colunas que interessam
colunas = [
    "id", "symbol", "name",
    "current_price", "market_cap",
    "total_volume", "price_change_percentage_24h",
    "high_24h", "low_24h"
]

df = df[colunas].copy()

# renomeia para ficar mais claro
df.columns = [
    "id", "simbolo", "nome",
    "preco_brl", "market_cap",
    "volume_24h", "variacao_24h_pct",
    "maxima_24h", "minima_24h"
]

print(f"Shape: {df.shape}")   # (linhas, colunas)
df.head(10)                   # mostra as 10 primeiras

Shape: (100, 9)


,id,simbolo,nome,preco_brl,market_cap,volume_24h,variacao_24h_pct,maxima_24h,minima_24h
0,bitcoin,btc,Bitcoin,366581.000000,7336167054748,1.970577e+11,-0.25153,371474.000000,361021.000000
1,ethereum,eth,Ethereum,11276.960000,1360919508250,8.784884e+10,-0.19332,11390.430000,11095.360000
2,tether,usdt,Tether,5.010000,922570752038,3.157205e+11,-1.79469,5.110000,5.010000
3,ripple,xrp,XRP,6.820000,418579464707,1.095693e+10,-1.68915,6.950000,6.720000
4,binancecoin,bnb,BNB,3053.890000,416349592080,5.111919e+09,-1.60199,3108.810000,3014.560000
5,usd-coin,usdc,USDC,5.010000,392951290344,6.749679e+10,-1.84296,5.110000,5.010000
6,solana,sol,Solana,427.020000,245123426152,1.770418e+10,-0.40200,434.530000,420.260000
7,tron,trx,TRON,1.590000,150609955762,2.339293e+09,-2.49723,1.630000,1.590000
8,figure-heloc,figr_heloc,Figure Heloc,5.120000,85086479683,1.158477e+08,-3.09679,5.280000,5.120000
9,dogecoin,doge,Dogecoin,0.473974,72870234989,6.358048e+09,-0.57508,0.482568,0.463877


In [5]:
# top 5 que mais subiram nas últimas 24h
print("TOP 5 maiores altas (24h):")
print(df.nlargest(5, "variacao_24h_pct")[["nome", "preco_brl", "variacao_24h_pct"]])

print()

# top 5 que mais caíram
print("TOP 5 maiores quedas (24h):")
print(df.nsmallest(5, "variacao_24h_pct")[["nome", "preco_brl", "variacao_24h_pct"]])

# salva em CSV
df.to_csv("moedas_coingecko.csv", index=False)
print("\nArquivo salvo: moedas_coingecko.csv")

TOP 5 maiores altas (24h):
                     nome   preco_brl  variacao_24h_pct
96                   Dash  218.320000          22.26970
86  Provenance Blockchain    0.055704           6.40667
84               Arbitrum    0.554715           5.39635
76                 Ethena    0.484171           4.10649
70                  Kaspa    0.169780           3.18843

TOP 5 maiores quedas (24h):
                       nome    preco_brl  variacao_24h_pct
37                Bittensor  1346.260000         -22.40812
36  World Liberty Financial     0.410369         -14.78962
99                    Siren     3.560000          -5.84085
91                     JUST     0.337078          -5.78105
83           Official Trump    14.440000          -5.05819

Arquivo salvo: moedas_coingecko.csv


In [6]:
from datetime import datetime

# adiciona a marca temporal de quando os dados foram coletados
df["coletado_em"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
df["fonte"]       = "coingecko_api"

# salva versão final com timestamp
df.to_csv("moedas_coingecko.csv", index=False)

print("Colunas do dataset final:")
print(df.dtypes)
print()
print(f"Linhas: {len(df)} | Colunas: {len(df.columns)}")
print()
print(df[["nome", "preco_brl", "variacao_24h_pct", "coletado_em"]].head(5))

Colunas do dataset final:
id                   object
simbolo              object
nome                 object
preco_brl           float64
market_cap            int64
volume_24h          float64
variacao_24h_pct    float64
maxima_24h          float64
minima_24h          float64
coletado_em          object
fonte                object
dtype: object

Linhas: 100 | Colunas: 11

       nome  preco_brl  variacao_24h_pct          coletado_em
0   Bitcoin  366581.00          -0.25153  2026-04-10 16:15:18
1  Ethereum   11276.96          -0.19332  2026-04-10 16:15:18
2    Tether       5.01          -1.79469  2026-04-10 16:15:18
3       XRP       6.82          -1.68915  2026-04-10 16:15:18
4       BNB    3053.89          -1.60199  2026-04-10 16:15:18


In [7]:
import os
print(os.getcwd())

C:\Users\junin


In [8]:
# força salvar como CSV de verdade
df.to_csv("moedas_coingecko.csv", index=False)

import os
print(os.path.abspath("moedas_coingecko.csv"))

C:\Users\junin\moedas_coingecko.csv


In [9]:
df.to_csv(r"C:\Users\junin\Downloads\moedas_coingecko.csv", index=False)
print("Salvo em Downloads!")

Salvo em Downloads!
